In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import sys
import os


# feature_path = os.path.abspath(r"C:\Users\alexg\OneDrive\Documents\NBA-Prop-Predictor")
feature_path = os.path.abspath('/Users/alexg/Documents/Documents/Prize-Picks-Prop-Predictor')
if feature_path not in sys.path:
    sys.path.append(feature_path)

from FEATURE_ENGINEERING.features import *
from PROPS_EV.calculateEVS import *
from PROPS_EV.backtest import *

### Load Model

In [2]:
model = joblib.load('Models/xgbModel.pkl')
features = joblib.load('Models/top_features.pkl')

### Load Data

In [3]:
pd.set_option('display.max_columns', None)
eplison = 0.000001

s25 = pd.read_csv('../DATA/CSV_FILES/TRAIN_DATA/PTS_TRAIN_25.csv')
s25['EXPECTED_USAGE_MIN'] = s25['USG_PCT_ROLLING_AVG_5'] * (s25['MIN_ROLLING_AVG_5'] + eplison)

s24 = pd.read_csv('../DATA/CSV_FILES/TRAIN_DATA/PTS_TRAIN_24.csv')
s24['EXPECTED_USAGE_MIN'] = s24['USG_PCT_ROLLING_AVG_5'] * (s24['MIN_ROLLING_AVG_5'] + eplison)
df = pd.concat([s25, s24]).sort_values(by='GAME_DATE')


date = '2025-04-11'
df = df[df['GAME_DATE'] < date]

dfsData = pd.read_csv('../DATA/CSV_FILES/BACKTEST_DATA/dfs_data.csv')
dfsData = dfsData[(dfsData['BOOKMAKER'] == 'prizepicks') & (dfsData['GAME_DATE'] == date) & (dfsData['CATEGORY'] == 'player_points')]
df.tail()

/var/folders/nw/9w4r5hrd05s122kt5hg_t8bw0000gn/T/ipykernel_44527/2323915385.py:15: DtypeWarning: Columns (11,12,14) have mixed types. Specify dtype option on import or set low_memory=False.
  dfsData = pd.read_csv('../DATA/CSV_FILES/BACKTEST_DATA/dfs_data.csv')


Unnamed: 0  Unnamed: 0.2        PLAYER_NAME  PLAYER_ID      MATCHUP  \
14774       14774       25591.0       Bobby Portis    1626171  MIL vs. NOP   
14773       14773       25593.0  Andre Jackson Jr.    1641748  MIL vs. NOP   
14772       14772       25614.0           AJ Green    1631260  MIL vs. NOP   
14783       14783       25596.0       Ryan Rollins    1631157  MIL vs. NOP   
14788       14788       25619.0    Jaden McDaniels    1630183    MIN @ MEM   

      TEAM_ABBREVIATION     TEAM_ID OPP_ABBREVIATION  HOME_GAME   GAME_ID  \
14774               MIL  1610612749              NOP          1  22401161   
14773               MIL  1610612749              NOP          1  22401161   
14772               MIL  1610612749              NOP          1  22401161   
14783               MIL  1610612749              NOP          1  22401161   
14788               MIN  1610612750              MEM          0  22401170   

        GAME_DATE WL  PTS  AST  REB  FGM  FGA  FG_PCT  FG3M  FG3A  FG3_PCT  \
14774  2025-04-10  W   14    0    8    5   10   0.500     2     4     0.50   
14773  2025-04-10  W    6    0    0    3    3   1.000     0     0      NaN   
14772  2025-04-10  W    3    3    3    1    4   0.250     1     4     0.25   
14783  2025-04-10  W   12    2    1    5    8   0.625     2     5     0.40   
14788  2025-04-10  W   10    1    5    4    7   0.571     2     4     0.50   

       FTM  FTA  FT_PCT  OREB  DREB  STL  BLK  TOV  PLUS_MINUS  FANTASY_PTS  \
14774    2    2     1.0     0     8    1    0    1          17         25.6   
14773    0    2     0.0     0     0    0    1    0           1          9.0   
14772    0    0     NaN     0     3    1    0    1          15         13.1   
14783    0    0     NaN     0     1    1    0    2           3         17.2   
14788    0    0     NaN     1     4    1    0    2          10         18.5   

       POINT_PER_SHOT       EFG START_POSITION  COMMENT  E_OFF_RATING  \
14774           1.287  0.600000            NaN      NaN         131.0   
14773           1.546  1.000000            NaN      NaN         106.2   
14772           0.750  0.375000            NaN      NaN         129.0   
14783           1.500  0.750000              G      NaN         121.0   
14788           1.429  0.714286              F      NaN         126.1   

       E_DEF_RATING  NET_RATING  OREB_PCT  DREB_PCT  REB_PCT  AST_PCT  \
14774          99.4        34.7     0.000     0.286    0.167    0.000   
14773         111.2        -0.7     0.000     0.000    0.000    0.000   
14772          95.1        34.6     0.000     0.120    0.068    0.158   
14783         119.5         9.9     0.000     0.056    0.027    0.125   
14788         114.4        13.2     0.033     0.138    0.085    0.043   

       EFG_PCT  AST_TOV  USG_PCT  TS_PCT  E_PACE    PACE    PIE  POSS  \
14774    0.600      0.0    0.231   0.643  106.78  106.91  0.152    49   
14773    1.000      0.0    0.211   0.773  118.63  115.60  0.123    18   
14772    0.375      3.0    0.102   0.375  110.58  111.79  0.044    46   
14783    0.750      1.0    0.227   0.750  103.87  105.15  0.106    42   
14788    0.714      0.5    0.118   0.714  110.04  107.40  0.062    67   

       PACE_PER40  E_USG_PCT    MIN   SPD  DIST  ORBC  DRBC  RBC  TCHS  SAST  \
14774       89.09      0.228  22.00  4.44  1.75     0    11   11    39     1   
14773       96.33      0.202   7.27  5.34  0.69     1     0    1    10     0   
14772       93.16      0.103  19.97  4.62  1.64     1     5    5    29     0   
14783       87.63      0.222  19.40  4.59  1.56     0     2    2    38     0   
14788       89.50      0.118  29.72  4.49  2.42     4     6    8    40     0   

       FTAST  PASS  CFGM  CFGA  CFG_PCT  UFGM  UFGA  UFG_PCT  DFGM  DFGA  \
14774      0    27     2     3    0.667     3     7    0.429     3     5   
14773      0     5     0     0    0.000     3     3    1.000     1     2   
14772      0    23     0     0    0.000     1     4    0.250     0     0   
14783      0    28     0     

In [4]:
backtestData = pd.read_csv('../DATA/CSV_FILES/BACKTEST_DATA/singleBookies.csv')
backtestData = backtestData[(backtestData['ODDS'] <= 200) & (backtestData['ODDS'] >= -200)]
singleBookies = backtestData[(backtestData['CATEGORY'] == 'points') & (backtestData['GAME_DATE'] == date)]
singleBookies

,NAME,CATEGORY,SIDE,BOOKMAKER,LINE,ODDS,fair_line,fair_odds,GAME_DATE
228880,Jaren Jackson Jr,points,under,draftkings,19.5,-105,19.5,-101,2025-04-11
228882,Jaren Jackson Jr,points,under,betrivers,19.5,-117,19.5,-101,2025-04-11
228885,Jaren Jackson Jr,points,over,draftkings,19.5,-125,19.5,101,2025-04-11
228887,Jaren Jackson Jr,points,over,betrivers,19.5,-114,19.5,101,2025-04-11
228903,Nickeil Alexander-Walker,points,over,fanduel,10.5,-130,10.5,-123,2025-04-11
...,...,...,...,...,...,...,...,...,...
234562,Jordan Goodwin,points,under,betmgm,3.5,-135,3.5,-115,2025-04-11
234571,Jeff Green,points,over,espnbet,7.5,-130,8.5,120,2025-04-11
234572,Jeff Green,points,under,espnbet,7.5,-110,8.5,-120,2025-04-11
234573,Jeff Green,points,over,espnbet,1.5,-130,1.5,-105,2025-04-11


### Top EVs for single bets

In [5]:
results = single_bet(
    data=df,
    bookmakers=singleBookies,
    model=model,
    features=features,
    stake=5,
    simulations=5000
)
results.sort_values(by='EV%', ascending=False).head(10)

Processing single bets...


,NAME,BOOKMAKER,CATEGORY,LINE,ODDS,SIDE,PREDICTION,RECOMMENDATION,OVER%,UNDER%,IMPLIED PROB,EV%,KELLY FULL,KELLY HALF,KELLY QUARTER,CONFIDENCE INTERVAL
1119,Aaron Gordon,fanduel,points,32.5,154,under,17.462973,1,0.003,0.997,0.394,153.14,0.99,0.50,0.25,"(7.2, 27.7)"
1673,Jalen Green,betmgm,points,14.5,185,over,22.060591,1,0.865,0.135,0.351,146.53,0.79,0.40,0.20,"(8.7, 35.3)"
1397,Oso Ighodaro,espnbet,points,2.5,160,over,7.245047,1,0.908,0.092,0.385,135.98,0.85,0.42,0.21,"(0.9, 15.3)"
835,Kyle Anderson,espnbet,points,0.5,135,over,6.132473,1,0.991,0.009,0.426,132.98,0.99,0.49,0.25,"(1.1, 12.0)"
469,Jrue Holiday,fanduel,points,3.5,130,over,14.400353,1,1.000,0.000,0.435,129.91,1.00,0.50,0.25,"(8.1, 20.7)"
1121,Aaron Gordon,espnbet,points,32.5,130,under,17.462973,1,0.001,0.999,0.435,129.72,1.00,0.50,0.25,"(7.2, 28.0)"
475,Derrick White,fanduel,points,3.5,132,over,18.329044,1,0.973,0.027,0.431,125.83,0.95,0.48,0.24,"(3.4, 35.0)"
1336,Jaden Springer,espnbet,points,2.5,125,over,13.075449,1,0.996,0.004,0.444,124.14,0.99,0.50,0.25,"(5.4, 20.6)"
300,Anthony Black,espnbet,points,2.5,130,over,9.641301,1,0.974,0.026,0.435,124.11,0.95,0.48,0.24,"(2.4, 17.1)"
714,Kevin Huerter,fanduel,points,3.5,132,over,16.600399,1,0.963,0.037,0.431,123.32,0.93,0.47,0.23,"(2.5, 33.1)"


### Top EVs for 2 leg bets

In [6]:
results = prizepickspairsEV(
    data=df,
    bookmakers=dfsData,
    model=model,
    features=features,
    stake=100,
    simulations=10000
)
results.sort_values(by='EV%', ascending=False).head(10).reset_index(drop=True)

Processing pairs...


,PLAYER 1,CATEGORY 1,BOOKMAKER 1,ODDS 1,LINE 1,SIDE 1,PREDICTION 1,MODEL_SIDE 1,OVER% 1,UNDER% 1,CONFIDENCE INTERVAL 1,PLAYER 2,CATEGORY 2,BOOKMAKER 2,ODDS 2,LINE 2,SIDE 2,PREDICTION 2,MODEL_SIDE 2,OVER% 2,UNDER% 2,CONFIDENCE INTERVAL 2,RECOMMENDED_TYPE,RECOMMENDATION,PROBABILITY,EV%,KELLY,KELLY FULL
0,Adem Bona,player_points,prizepicks,-137,13.5,over,10.85,UNDER,0.093,0.907,"(6.9, 14.8)",Jrue Holiday,player_points,prizepicks,-137,9.5,over,14.40,OVER,0.931,0.069,"(8.1, 20.6)",UNDER/OVER,0,0.8442,1.533,0.766,0.766
1,Goga Bitadze,player_points,prizepicks,-137,12.5,over,4.75,UNDER,0.102,0.898,"(0.4, 16.3)",Jrue Holiday,player_points,prizepicks,-137,9.5,over,14.40,OVER,0.931,0.069,"(8.1, 20.6)",UNDER/OVER,1,0.8362,1.509,0.754,0.754
2,Cory Joseph,player_points,prizepicks,-137,8.0,over,5.50,UNDER,0.108,0.892,"(1.6, 9.3)",Jrue Holiday,player_points,prizepicks,-137,9.5,over,14.40,OVER,0.931,0.069,"(8.1, 20.6)",UNDER/OVER,0,0.8304,1.491,0.746,0.746
3,Adem Bona,player_points,prizepicks,-137,13.5,over,10.85,UNDER,0.093,0.907,"(6.9, 14.8)",Goga Bitadze,player_points,prizepicks,-137,12.5,over,4.75,UNDER,0.102,0.898,"(0.4, 16.3)",UNDER/UNDER,0,0.8145,1.443,0.722,0.722
4,Jrue Holiday,player_points,prizepicks,-137,9.5,over,14.40,OVER,0.931,0.069,"(8.1, 20.6)",Karlo Matković,player_points,prizepicks,-137,14.5,over,12.26,UNDER,0.130,0.870,"(8.3, 16.2)",OVER/UNDER,0,0.8103,1.431,0.716,0.716
5,Adem Bona,player_points,prizepicks,-137,13.5,over,10.85,UNDER,0.093,0.907,"(6.9, 14.8)",Cory Joseph,player_points,prizepicks,-137,8.0,over,5.50,UNDER,0.108,0.892,"(1.6, 9.3)",UNDER/UNDER,0,0.8088,1.426,0.713,0.713
6,Adem Bona,player_points,prizepicks,-137,13.5,over,10.85,UNDER,0.093,0.907,"(6.9, 14.8)",Al Horford,player_points,prizepicks,-137,9.0,over,13.77,OVER,0.883,0.117,"(5.8, 21.8)",UNDER/OVER,0,0.8008,1.402,0.701,0.701
7,Jrue Holiday,player_points,prizepicks,-137,9.5,over,14.40,OVER,0.931,0.069,"(8.1, 20.6)",Mike Conley,player_points,prizepicks,-137,7.5,over,11.32,OVER,0.859,0.141,"(4.4, 18.1)",OVER/OVER,0,0.7998,1.399,0.700,0.700
8,Jrue Holiday,player_points,prizepicks,-137,9.5,over,14.40,OVER,0.931,0.069,"(8.1, 20.6)",Lonnie Walker IV,player_points,prizepicks,-137,20.0,over,14.60,UNDER,0.146,0.854,"(4.7, 24.6)",OVER/UNDER,1,0.7954,1.386,0.693,0.693
9,Jrue Holiday,player_points,prizepicks,-137,9.5,over,14.40,OVER,0.931,0.069,"(8.1, 20.6)",Quentin Grimes,player_points,prizepicks,-137,24.5,over,20.02,UNDER,0.146,0.854,"(11.5, 28.6)",OVER/UNDER,0,0.7952,1.386,0.693,0.693
